In [1]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import joblib

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import LoanFeatureEngineer

print("Libraries imported successfully!")
print("Project root:", PROJECT_ROOT)


Libraries imported successfully!
Project root: c:\Users\Admin\Desktop\python_new\github\week-2-weekly-assessment


In [2]:

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "lending_club_loans.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()


Dataset loaded successfully!
Dataset shape: (396030, 27)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,address
0,10000.0,36 months,11.44,329.48,B,B4,Marketing,10+ years,RENT,117000.0,...,16.0,0.0,36369.0,41.8,25.0,w,INDIVIDUAL,0.0,0.0,"0174 Michelle Gateway\r\nMendozaberg, OK 22690"
1,8000.0,36 months,11.99,265.68,B,B5,Credit analyst,4 years,MORTGAGE,65000.0,...,17.0,0.0,20131.0,53.3,27.0,f,INDIVIDUAL,3.0,0.0,"1076 Carney Fort Apt. 347\r\nLoganmouth, SD 05113"
2,15600.0,36 months,10.49,506.97,B,B3,Statistician,< 1 year,RENT,43057.0,...,13.0,0.0,11987.0,92.2,26.0,f,INDIVIDUAL,0.0,0.0,"87025 Mark Dale Apt. 269\r\nNew Sabrina, WV 05113"
3,7200.0,36 months,6.49,220.65,A,A2,Client Advocate,6 years,RENT,54000.0,...,6.0,0.0,5472.0,21.5,13.0,f,INDIVIDUAL,0.0,0.0,"823 Reid Ford\r\nDelacruzside, MA 00813"
4,24375.0,60 months,17.27,609.33,C,C5,Destiny Management Inc.,9 years,MORTGAGE,55000.0,...,13.0,0.0,24584.0,69.8,43.0,f,INDIVIDUAL,1.0,0.0,"679 Luna Roads\r\nGreggshire, VA 11650"


In [3]:

df["default"] = (
    df["loan_status"] == "Charged Off"
).astype(int)

print("Target distribution:")
print(df["default"].value_counts())

print("\nTarget proportions:")
print(df["default"].value_counts(normalize=True) * 100)


Target distribution:
default
0    318357
1     77673
Name: count, dtype: int64

Target proportions:
default
0    80.387092
1    19.612908
Name: proportion, dtype: float64


In [4]:

feature_cols = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "issue_d",
    "earliest_cr_line",
    "dti",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "initial_list_status",
    "application_type",
    "mort_acc",
    "pub_rec_bankruptcies"
]

X = df[feature_cols].copy()
y = df["default"].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

X.head()


Feature shape: (396030, 23)
Target shape: (396030,)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,dti,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies
0,10000.0,36 months,11.44,329.48,B,B4,10+ years,RENT,117000.0,Not Verified,...,26.24,16.0,0.0,36369.0,41.8,25.0,w,INDIVIDUAL,0.0,0.0
1,8000.0,36 months,11.99,265.68,B,B5,4 years,MORTGAGE,65000.0,Not Verified,...,22.05,17.0,0.0,20131.0,53.3,27.0,f,INDIVIDUAL,3.0,0.0
2,15600.0,36 months,10.49,506.97,B,B3,< 1 year,RENT,43057.0,Source Verified,...,12.79,13.0,0.0,11987.0,92.2,26.0,f,INDIVIDUAL,0.0,0.0
3,7200.0,36 months,6.49,220.65,A,A2,6 years,RENT,54000.0,Not Verified,...,2.60,6.0,0.0,5472.0,21.5,13.0,f,INDIVIDUAL,0.0,0.0
4,24375.0,60 months,17.27,609.33,C,C5,9 years,MORTGAGE,55000.0,Verified,...,33.95,13.0,0.0,24584.0,69.8,43.0,f,INDIVIDUAL,1.0,0.0


In [5]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


Training: (237618, 23)
Validation: (79206, 23)
Testing: (79206, 23)


In [6]:

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


Numerical features:
['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies']

Categorical features:
['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'issue_d', 'earliest_cr_line', 'initial_list_status', 'application_type']


In [7]:

missing_values = X_train.isnull().sum()

missing_values = missing_values[
    missing_values > 0
].sort_values(ascending=False)

print("Features with missing values:")
print(missing_values)


Features with missing values:
mort_acc                22622
emp_length              10978
pub_rec_bankruptcies      314
revol_util                161
dtype: int64


In [8]:

numeric_imputer = SimpleImputer(
    strategy="median"
)

categorical_imputer = SimpleImputer(
    strategy="most_frequent"
)

numeric_imputed = numeric_imputer.fit_transform(
    X_train[numeric_features]
)

categorical_imputed = categorical_imputer.fit_transform(
    X_train[categorical_features]
)

print("Numerical missing values after imputation:",
      np.isnan(numeric_imputed).sum())

print("Categorical missing values after imputation:",
      pd.isna(categorical_imputed).sum())


Numerical missing values after imputation: 0
Categorical missing values after imputation: 0


In [9]:

knn_columns = [
    "loan_amnt",
    "annual_inc",
    "int_rate",
    "dti",
    "revol_bal",
    "revol_util",
    "installment",
    "open_acc"
]

knn_sample = X_train[knn_columns].sample(
    n=min(1000, len(X_train)),
    random_state=42
)

knn_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("imputer", KNNImputer(
        n_neighbors=5,
        weights="distance"
    ))
])

knn_imputed = knn_pipeline.fit_transform(knn_sample)

print("Original sample shape:", knn_sample.shape)
print("KNN imputed shape:", knn_imputed.shape)

print(
    "Missing values before:",
    knn_sample.isnull().sum().sum()
)

print(
    "Missing values after:",
    np.isnan(knn_imputed).sum()
)


Original sample shape: (1000, 8)
KNN imputed shape: (1000, 8)
Missing values before: 1
Missing values after: 0


In [10]:

ordinal_features = [
    "grade",
    "sub_grade"
]

onehot_features = [
    "term",
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    "initial_list_status",
    "application_type"
]

print("Ordinal features:", ordinal_features)
print("One-hot features:", onehot_features)


Ordinal features: ['grade', 'sub_grade']
One-hot features: ['term', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'application_type']


In [11]:

grade_categories = [
    "A", "B", "C", "D", "E", "F", "G"
]

sub_grade_categories = [
    f"{grade}{number}"
    for grade in "ABCDEFG"
    for number in range(1, 6)
]

ordinal_categories = [
    grade_categories,
    sub_grade_categories
]

ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

print("Grade categories:", grade_categories)
print("Sub-grade categories:", sub_grade_categories)


Grade categories: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Sub-grade categories: ['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B2', 'B3', 'B4', 'B5', 'C1', 'C2', 'C3', 'C4', 'C5', 'D1', 'D2', 'D3', 'D4', 'D5', 'E1', 'E2', 'E3', 'E4', 'E5', 'F1', 'F2', 'F3', 'F4', 'F5', 'G1', 'G2', 'G3', 'G4', 'G5']


In [12]:

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore"
)

onehot_result = onehot_encoder.fit_transform(
    X_train[onehot_features]
)

ordinal_result = ordinal_encoder.fit_transform(
    X_train[ordinal_features]
)

print("OneHot encoded shape:", onehot_result.shape)
print("Ordinal encoded shape:", ordinal_result.shape)


OneHot encoded shape: (237618, 42)
Ordinal encoded shape: (237618, 2)


In [13]:

feature_engineer = LoanFeatureEngineer()

engineered_sample = feature_engineer.fit_transform(
    X_train.head(5)
)

print("Original features:")
print(X_train.head(5).columns.tolist())

print("\nEngineered features:")
print(engineered_sample.columns.tolist())

engineered_sample.head()


Original features:
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'issue_d', 'earliest_cr_line', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'mort_acc', 'pub_rec_bankruptcies']

Engineered features:
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'mort_acc', 'pub_rec_bankruptcies', 'issue_year', 'issue_month', 'credit_history_months', 'income_to_loan_ratio', 'log_annual_inc']


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,issue_year,issue_month,credit_history_months,income_to_loan_ratio,log_annual_inc
377623,12000.0,36 months,7.68,374.29,A,A2,5 years,MORTGAGE,69996.0,Verified,...,35.0,f,INDIVIDUAL,NaN,0.0,2009,4,96,5.833000,11.156208
161240,18450.0,60 months,14.99,438.83,C,C5,2 years,RENT,80000.0,Source Verified,...,38.0,w,INDIVIDUAL,2.0,0.0,2014,12,226,4.336043,11.289794
163721,10000.0,36 months,9.17,318.79,B,B2,5 years,MORTGAGE,50000.0,Source Verified,...,20.0,w,INDIVIDUAL,3.0,0.0,2016,1,158,5.000000,10.819798
158117,30000.0,60 months,19.29,783.02,E,E4,9 years,MORTGAGE,80000.0,Verified,...,13.0,f,INDIVIDUAL,NaN,0.0,2011,8,102,2.666667,11.289794
137235,5000.0,36 months,17.57,179.69,D,D4,10+ years,MORTGAGE,70000.0,Verified,...,37.0,w,INDIVIDUAL,4.0,1.0,2014,7,226,14.000000,11.156265


In [14]:
[
    "issue_year",
    "issue_month",
    "credit_history_months",
    "income_to_loan_ratio",
    "log_annual_inc"
]

['issue_year',
 'issue_month',
 'credit_history_months',
 'income_to_loan_ratio',
 'log_annual_inc']

In [15]:

engineered_numeric_features = numeric_features + [
    "issue_year",
    "issue_month",
    "credit_history_months",
    "income_to_loan_ratio",
    "log_annual_inc"
]

print("Final numerical features:")
print(engineered_numeric_features)

print("\nOrdinal features:")
print(ordinal_features)

print("\nOne-hot features:")
print(onehot_features)


Final numerical features:
['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'mort_acc', 'pub_rec_bankruptcies', 'issue_year', 'issue_month', 'credit_history_months', 'income_to_loan_ratio', 'log_annual_inc']

Ordinal features:
['grade', 'sub_grade']

One-hot features:
['term', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'application_type']


In [16]:

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median"
    )),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

print("All preprocessing transformers created!")


All preprocessing transformers created!


In [17]:

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            engineered_numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            onehot_features
        ),
        (
            "ordinal",
            ordinal_transformer,
            ordinal_features
        )
    ],
    remainder="drop"
)

print("ColumnTransformer created successfully!")


ColumnTransformer created successfully!


In [18]:

loan_pipeline = Pipeline(steps=[
    (
        "feature_engineering",
        LoanFeatureEngineer()
    ),
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])

print("Complete ML pipeline created!")


Complete ML pipeline created!


In [19]:

loan_pipeline.fit(
    X_train,
    y_train
)

print("Pipeline training completed!")


Pipeline training completed!


In [20]:

val_predictions = loan_pipeline.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_predictions
)

print("Validation Accuracy:", val_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_predictions,
        target_names=[
            "Fully Paid",
            "Charged Off"
        ]
    )
)


Validation Accuracy: 0.8062899275307426

Classification Report:
              precision    recall  f1-score   support

  Fully Paid       0.82      0.98      0.89     63671
 Charged Off       0.54      0.09      0.15     15535

    accuracy                           0.81     79206
   macro avg       0.68      0.53      0.52     79206
weighted avg       0.76      0.81      0.75     79206



In [21]:

MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH = MODEL_DIR / "loan_default_pipeline.joblib"

joblib.dump(
    loan_pipeline,
    MODEL_PATH
)

print("Pipeline saved successfully!")
print("Saved location:", MODEL_PATH)


Pipeline saved successfully!
Saved location: c:\Users\Admin\Desktop\python_new\github\week-2-weekly-assessment\models\loan_default_pipeline.joblib


In [22]:

loaded_pipeline = joblib.load(
    MODEL_PATH
)

print("Pipeline loaded successfully!")


Pipeline loaded successfully!


In [23]:

new_raw_data = X_test.sample(
    n=3,
    random_state=42
).copy()

print("Three raw input records:")
display(new_raw_data)


Three raw input records:


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,dti,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies
280845,4425.0,36 months,17.27,158.36,D,D3,2 years,MORTGAGE,69000.0,Source Verified,...,18.47,11.0,0.0,6534.0,66.0,31.0,f,INDIVIDUAL,2.0,0.0
65397,5675.0,36 months,16.99,202.31,D,D3,2 years,OWN,24000.0,Source Verified,...,29.90,10.0,1.0,2518.0,24.0,36.0,w,INDIVIDUAL,0.0,0.0
11578,25000.0,60 months,22.45,696.89,E,E5,10+ years,MORTGAGE,134000.0,Verified,...,24.05,15.0,1.0,9970.0,57.3,53.0,w,INDIVIDUAL,4.0,1.0


In [24]:

predictions = loaded_pipeline.predict(
    new_raw_data
)

probabilities = loaded_pipeline.predict_proba(
    new_raw_data
)

prediction_results = new_raw_data.copy()

prediction_results["predicted_default"] = predictions

prediction_results["default_probability"] = probabilities[:, 1]

prediction_results["predicted_status"] = (
    prediction_results["predicted_default"]
    .map({
        0: "Fully Paid",
        1: "Charged Off"
    })
)

display(
    prediction_results[
        [
            "loan_amnt",
            "annual_inc",
            "predicted_default",
            "default_probability",
            "predicted_status"
        ]
    ]
)


,loan_amnt,annual_inc,predicted_default,default_probability,predicted_status
280845,4425.0,69000.0,0,0.200596,Fully Paid
65397,5675.0,24000.0,0,0.340212,Fully Paid
11578,25000.0,134000.0,0,0.370350,Fully Paid
